# steam_stratified_sample.csv 생성
## 조건
- 출시연도: 2023 ~ 2025년
- 리뷰 수: 10개 이상
- 타입: game
- 총 표본: 160개
- F2P 비율: 모집단 비율에 맞게 보정
- 2026년 게임 제외


## 1. 라이브러리 로드

In [28]:
import pandas as pd
import numpy as np
import ast
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
print("완료")


완료


## 2. 데이터 로드 및 전처리

In [29]:
df = pd.read_csv("../../../data/raw/steam_indie_list.csv")

df['total_reviews'] = df['positive'] + df['negative']
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')

def parse_owners_lower(s):
    try:
        return int(s.split('..')[0].strip().replace(',', ''))
    except:
        return 0

def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except:
        return []

df['owners_lower'] = df['owners'].apply(parse_owners_lower)
df['genres_list']  = df['genres'].apply(parse_genres)
df['is_f2p']       = df['genres_list'].apply(lambda gl: 'Free To Play' in gl)

print(f"shape: {df.shape}")
print("전처리 완료")


shape: (61266, 16)
전처리 완료


## 3. 필터 적용

In [30]:
df_f = df[
    (df['total_reviews'] >= 10) &
    (df['release_date'].dt.year >= 2023) &
    (df['release_date'].dt.year < 2026) &
    (df['type'] == 'game')
].copy()

print(f"필터 전: {len(df):,}개")
print(f"필터 후: {len(df_f):,}개")
print(f"\n출시연도 분포:")
print(df_f['release_date'].dt.year.value_counts().sort_index())


필터 전: 61,266개
필터 후: 11,344개

출시연도 분포:
release_date
2023    4100
2024    4947
2025    2297
Name: count, dtype: int64


## 4. Wilson Score 경계값 계산

In [31]:
Z = 1.96

def wilson_margin(n):
    p     = 0.5
    denom = 1 + Z**2 / n
    return (Z / denom) * np.sqrt(p*(1-p)/n + Z**2/(4*n**2)) * 100

def find_n_for_margin(target_pct):
    for n in range(1, 10000):
        if wilson_margin(n) <= target_pct:
            return n
    return 10000

LOW_BOUNDARY  = find_n_for_margin(15)   # 저→중 경계
HIGH_BOUNDARY = find_n_for_margin(5)    # 중→고 경계

print(f"저신뢰 (low) : 리뷰 10 ~ {LOW_BOUNDARY-1}개  (±15% 초과)")
print(f"중신뢰 (mid) : 리뷰 {LOW_BOUNDARY} ~ {HIGH_BOUNDARY-1}개  (±5~15%)")
print(f"고신뢰 (high): 리뷰 {HIGH_BOUNDARY}개 이상  (±5% 이하)")


저신뢰 (low) : 리뷰 10 ~ 38개  (±15% 초과)
중신뢰 (mid) : 리뷰 39 ~ 380개  (±5~15%)
고신뢰 (high): 리뷰 381개 이상  (±5% 이하)


## 5. 층 할당

In [32]:
def assign_stratum(row):
    scale = ('large' if row['owners_lower'] >= 200_000
             else 'mid' if row['owners_lower'] >= 20_000
             else 'small')
    trust = ('high' if row['total_reviews'] >= HIGH_BOUNDARY
             else 'mid' if row['total_reviews'] >= LOW_BOUNDARY
             else 'low')
    return f'{scale}_{trust}'

df_f['stratum'] = df_f.apply(assign_stratum, axis=1)

print("=== 층별 모집단 크기 ===")
print(df_f['stratum'].value_counts().sort_index())


=== 층별 모집단 크기 ===
stratum
large_high     407
large_low       18
large_mid       24
mid_high      1079
mid_low        583
mid_mid        848
small_high     193
small_low     4890
small_mid     3302
Name: count, dtype: int64


## 6. 층화 추출

In [33]:
SAMPLE_PLAN = {
    'large_high': 30,
    'large_mid' : 15,
    'large_low' : 10,
    'mid_high'  : 25,
    'mid_mid'   : 20,
    'mid_low'   : 10,
    'small_high': 20,
    'small_mid' : 15,
    'small_low' : 15,
}

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sampled_frames = []

print(f"{'층':<15} {'모집단':>7}  {'추출':>5}  {'추출률':>7}")
print('-' * 42)

for stratum, n in SAMPLE_PLAN.items():
    pool     = df_f[df_f['stratum'] == stratum]
    actual_n = min(n, len(pool))
    sample   = pool.sample(n=actual_n, random_state=RANDOM_SEED)
    sampled_frames.append(sample)
    rate = actual_n / len(pool) * 100 if len(pool) > 0 else 0
    print(f"{stratum:<15} {len(pool):>7}  {actual_n:>5}  {rate:>6.1f}%")

df_sample = pd.concat(sampled_frames).reset_index(drop=True)
print('-' * 42)
print(f"{'총계':<15} {len(df_f):>7}  {len(df_sample):>5}")


층                   모집단     추출      추출률
------------------------------------------
large_high          407     30     7.4%
large_mid            24     15    62.5%
large_low            18     10    55.6%
mid_high           1079     25     2.3%
mid_mid             848     20     2.4%
mid_low             583     10     1.7%
small_high          193     20    10.4%
small_mid          3302     15     0.5%
small_low          4890     15     0.3%
------------------------------------------
총계                11344    160


## 7. F2P 비율 보정

In [34]:
F2P_RATE   = df_f['is_f2p'].mean()
F2P_TARGET = round(len(df_sample) * F2P_RATE)
current_f2p = df_sample['is_f2p'].sum()

print(f"모집단 F2P 비율 : {F2P_RATE:.1%}")
print(f"조정 전 F2P 수  : {current_f2p}개 ({current_f2p/len(df_sample):.1%})")
print(f"목표 F2P 수     : {F2P_TARGET}개")

excess = current_f2p - F2P_TARGET

if excess > 0:
    f2p_idx  = df_sample[df_sample['is_f2p']].index.tolist()
    np.random.seed(RANDOM_SEED)
    drop_idx = np.random.choice(f2p_idx, size=excess, replace=False)
    df_sample = df_sample.drop(index=drop_idx).reset_index(drop=True)

    used_appids    = set(df_sample['appid'])
    non_f2p_pool   = df_f[(~df_f['is_f2p']) & (~df_f['appid'].isin(used_appids))]
    fill_samples   = []
    current_counts = df_sample['stratum'].value_counts().to_dict()

    for i in range(excess):
        best_stratum = max(
            SAMPLE_PLAN.keys(),
            key=lambda s: len(df_f[df_f['stratum'] == s]) - current_counts.get(s, 0)
        )
        pool_fill = non_f2p_pool[
            (non_f2p_pool['stratum'] == best_stratum) &
            (~non_f2p_pool['appid'].isin(used_appids))
        ]
        if len(pool_fill) > 0:
            picked = pool_fill.sample(1, random_state=RANDOM_SEED + i)
            fill_samples.append(picked)
            used_appids.add(picked['appid'].values[0])
            current_counts[best_stratum] = current_counts.get(best_stratum, 0) + 1

    if fill_samples:
        df_sample = pd.concat([df_sample] + fill_samples).reset_index(drop=True)

final_f2p = df_sample['is_f2p'].sum()
print(f"\n조정 후 F2P 수  : {final_f2p}개 ({final_f2p/len(df_sample):.1%})")
print(f"최종 표본 수    : {len(df_sample)}개")


모집단 F2P 비율 : 4.4%
조정 전 F2P 수  : 8개 (5.0%)
목표 F2P 수     : 7개

조정 후 F2P 수  : 7개 (4.4%)
최종 표본 수    : 160개


## 8. 검증 및 저장

In [35]:
print("=== 층별 최종 구성 ===")
print(df_sample['stratum'].value_counts().sort_index())

print("\n=== 출시연도 분포 ===")
print(df_sample['release_date'].dt.year.value_counts().sort_index())

OUT_COLS = [
    'appid', 'name_store', 'release_date', 'genres',
    'owners', 'owners_lower', 'positive', 'negative',
    'total_reviews', 'price_spy', 'ccu', 'developers',
    'stratum', 'is_f2p'
]

df_sample[OUT_COLS].to_csv("../../../data/processed/steam_stratified_sample_v4.csv", index=False)
print(f"\n저장 완료 → steam_stratified_sample_v4.csv ({len(df_sample)}개)")
df_sample[OUT_COLS].head()


=== 층별 최종 구성 ===
stratum
large_high    29
large_low     10
large_mid     15
mid_high      25
mid_low       10
mid_mid       20
small_high    20
small_low     16
small_mid     15
Name: count, dtype: int64

=== 출시연도 분포 ===
release_date
2023    68
2024    71
2025    21
Name: count, dtype: int64

저장 완료 → steam_stratified_sample_v4.csv (160개)


,appid,name_store,release_date,genres,owners,owners_lower,positive,negative,total_reviews,price_spy,ccu,developers,stratum,is_f2p
0,1432860,Sun Haven,2023-03-10,"['Adventure', 'Casual', 'Indie', 'RPG', 'Simul...","500,000 .. 1,000,000",500000,18523,3933,22456,2499,653,Pixel Sprout Studios,large_high,False
1,1473350,(the) Gnorp Apologue,2023-12-14,"['Casual', 'Indie', 'Simulation', 'Strategy']","200,000 .. 500,000",200000,7849,294,8143,699,289,Myco,large_high,False
2,1993150,轮回修仙路,2023-06-19,"['Adventure', 'Indie', 'RPG', 'Simulation']","200,000 .. 500,000",200000,2355,510,2865,1699,14,烟水寒工作室,large_high,False
3,2527500,MiSide,2024-12-10,"['Adventure', 'Indie', 'RPG', 'Simulation']","1,000,000 .. 2,000,000",1000000,108883,2204,111087,1499,631,AIHASTO,large_high,False
4,1169040,Necesse,2025-10-16,"['Action', 'Adventure', 'Indie', 'RPG']","1,000,000 .. 2,000,000",1000000,15988,1083,17071,974,507,Fair Games ApS,large_high,False
